# 08 · Queen Transformer and allocation policy

**Status: UNEXECUTED FIRST DRAFT.** This notebook contains working implementations to inspect and refine when run. No trained capability or benchmark result is asserted.

**Purpose:** Condition goals, action preferences and Worker/Scout allocation on recent observations and memory, while external code enforces legality and finite budgets.

**Inputs:** Notebook 07 role definitions; semantic history [B,H,D], memory [B,M,D], action mask [B,8].

**Outputs:** QueenController with goal, role mixture, halt, action, click-coordinate and value outputs.

**Expected runtime:** Under a minute for metadata checks; full forward and budget validation in 09. These are planning estimates; measure on this instance.

**Checkpoint directory:** `checkpoints/<run_id>/` under the project root. Definitions in this notebook are included in every source manifest.

**Exact resume:** Restart the AHR v2 XL kernel and Run All. This notebook defines components and runs bounded checks; it does not start a long training job. Training continuation is handled in 10 and 11 from the latest valid checkpoint.

The configuration remains the full XL design (`d_model=1024` by default). Any explicitly identified synthetic fixture verifies plumbing only and is not a smaller AHR training configuration.


## Load explicit notebook dependencies
Only tagged definitions are loaded; dependency execution cells are not run.

In [ ]:
from pathlib import Path
import json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'notebooks' / '00_setup_and_hardware.ipynb').exists()), None)
if ROOT is None:
    raise RuntimeError('Open a kernel in /workspace/arc-agi-3-ahrv2 or its notebooks directory.')
_nb = json.loads((ROOT / 'notebooks' / '00_setup_and_hardware.ipynb').read_text())
for _cell in _nb['cells']:
    if _cell['cell_type'] == 'code' and 'ahr-export' in _cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_cell['source']), '00_setup_and_hardware.ipynb', 'exec'), globals())
load_exports([7], globals())

## Queen controller

History and memory are distinct token types. Positions encode finite history order. The controller can request halting, but cannot override the public action mask, provenance labels or hard planning budget. Its goal is a learned latent target, not a textual game rule.

In [ ]:
class QueenController(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = copy.deepcopy(cfg)
        m, r = cfg['model'], cfg['reasoning']
        d = int(m['d_model'])
        self.history_length = int(r['history_length'])
        self.memory_capacity = int(r['memory_capacity'])
        self.summary_token = nn.Parameter(torch.randn(1, 1, d)*0.02)
        self.type_embeddings = nn.Parameter(torch.randn(3, d)*0.02)
        self.history_positions = nn.Parameter(torch.randn(self.history_length, d)*0.02)
        self.memory_positions = nn.Parameter(torch.randn(self.memory_capacity, d)*0.02)
        layer = nn.TransformerEncoderLayer(d, int(m['n_heads']), 4*d, float(m['dropout']),
                                          activation='gelu', batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, int(m['queen_layers']), enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.goal_head = nn.Linear(d, d)
        self.role_head = nn.Linear(d, 2)
        self.halt_head = nn.Linear(d, 1)
        self.action_head = nn.Linear(d, int(m['action_dim']))
        self.value_head = nn.Linear(d, 1)
        self.click_head = nn.Linear(d, 2*int(m['grid_size']))
        self.grid_size = int(m['grid_size'])

    def forward(self, history, memory, mask):
        if history.ndim != 3 or memory.ndim != 3 or history.shape[0] != memory.shape[0]:
            raise ValueError('Queen expects batched semantic history and memory.')
        if history.shape[1] < 1:
            raise ValueError('Queen needs a current observation.')
        history, memory = history[:, -self.history_length:], memory[:, :self.memory_capacity]
        history_tokens = history + self.type_embeddings[0] + self.history_positions[:history.shape[1]]
        memory_tokens = memory + self.type_embeddings[1] + self.memory_positions[:memory.shape[1]]
        summary = self.summary_token.expand(history.shape[0], -1, -1) + self.type_embeddings[2]
        tokens = torch.cat((summary, history_tokens, memory_tokens), dim=1)
        hidden = self.norm(self.transformer(tokens)[:, 0])
        goal = torch.tanh(self.goal_head(hidden))*float(self.cfg['reasoning']['latent_clip'])
        roles = self.role_head(hidden).float().softmax(dim=-1)
        if self.cfg['reasoning']['queen_mode'] == 'fixed':
            # Ablation removes adaptive goal and allocation while retaining comparable parameters.
            goal = history[:, -1]
            roles = torch.full_like(roles, 0.5)
        elif self.cfg['reasoning']['queen_mode'] != 'learned':
            raise ValueError('queen_mode must be learned or fixed')
        return {'goal': goal, 'worker_weight': roles[:, 0], 'scout_weight': roles[:, 1],
                'halt_logit': self.halt_head(hidden).squeeze(-1),
                'action_logits': mask_logits(self.action_head(hidden), mask),
                'value': self.value_head(hidden).squeeze(-1),
                'click_logits': self.click_head(hidden).reshape(history.shape[0], 2, self.grid_size)}

## Run All: bounded validation

This checks the full Queen parameter count and the explicit controller configuration. Its numerical outputs are checked with the integrated XL model in 09.

In [ ]:
cfg = load_config()
with torch.device('meta'):
    queen_metadata = QueenController(cfg)
print({'queen_parameters': sum(p.numel() for p in queen_metadata.parameters()),
       'queen_layers': cfg['model']['queen_layers'], 'mode': cfg['reasoning']['queen_mode'],
       'history_capacity': cfg['reasoning']['history_length'], 'memory_capacity': cfg['reasoning']['memory_capacity']})
del queen_metadata
assert cfg['reasoning']['queen_mode'] in ('learned', 'fixed')